In [1]:
import pandas as pd
import numpy as np

# Load model-ready dataset
df = pd.read_csv(
    "../data/processed/transactions_model_ready.csv",
    low_memory=False
)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

df.head()

Rows: 999,891
Columns: 26


,transaction_id,user_id,timestamp,amount,merchant_id,merchant_category,location,device_id,ip_address,payment_method,...,is_fraud,flag_missing_amount,flag_negative_amount,flag_missing_user,flag_missing_merchant,flag_negative_distance,flag_duplicate_transaction,flag_invalid_timestamp,flag_unknown_category,data_quality_issue
0,TXN000987232,USER001150,2026-06-22 16:50:47,1059.70,MERCHANT002035,retail,Delhi,DEVICE003339,192.0.2.59,upi,...,0.0,False,False,False,False,False,False,False,False,False
1,TXN000079955,USER000731,2026-07-17 13:55:11,850.27,MERCHANT003234,travel,Pune,DEVICE001104,192.0.2.195,upi,...,0.0,False,False,False,False,False,False,False,False,False
2,TXN000567131,USER010498,2026-06-09 14:16:38,1532.67,MERCHANT002779,restaurant,Chennai,DEVICE003272,192.0.2.208,card,...,1.0,False,False,False,False,False,False,False,False,False
3,TXN000500892,USER009411,2026-04-11 04:34:26,2590.01,MERCHANT002694,healthcare,Delhi,DEVICE009703,192.0.2.220,upi,...,0.0,False,False,False,False,False,False,False,False,False
4,TXN000055400,USER011446,2026-04-16 08:27:18,4413.39,MERCHANT001817,electronics,Kochi,DEVICE005030,192.0.2.167,net_banking,...,1.0,False,False,False,False,False,False,False,False,False


In [2]:
print(df.dtypes)

transaction_id                    str
user_id                           str
timestamp                         str
amount                        float64
merchant_id                       str
merchant_category                 str
location                          str
device_id                         str
ip_address                        str
payment_method                    str
account_age                     int64
transaction_count_24h         float64
avg_transaction_amount        float64
distance_from_home            float64
is_new_device                 float64
is_new_location               float64
is_fraud                      float64
flag_missing_amount              bool
flag_negative_amount             bool
flag_missing_user                bool
flag_missing_merchant            bool
flag_negative_distance           bool
flag_duplicate_transaction       bool
flag_invalid_timestamp           bool
flag_unknown_category            bool
data_quality_issue               bool
dtype: objec

In [3]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

print(df["timestamp"].dtype)

datetime64[us]


In [4]:
df = df.sort_values(
    ["user_id", "timestamp"]
).reset_index(drop=True)

In [5]:
df = df.sort_values(
    ["user_id", "timestamp"]
).reset_index(drop=True)

**<h1>Feature Engineering</H1>**

In [6]:
# ==========================================
# Temporal Features
# ==========================================

df["hour"] = df["timestamp"].dt.hour

df["day_of_week"] = df["timestamp"].dt.dayofweek

df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(int)

df["day"] = df["timestamp"].dt.day

df["month"] = df["timestamp"].dt.month

In [21]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/processed/transactions_model_ready.csv",
    low_memory=False
)

df["timestamp"] = pd.to_datetime(df["timestamp"])

df = df.sort_values(
    ["user_id", "timestamp"]
).reset_index(drop=True)

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 999891
Columns: 26


In [22]:
# ==========================================
# Temporal Features
# ==========================================

df["hour"] = df["timestamp"].dt.hour

df["day_of_week"] = df["timestamp"].dt.dayofweek

df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(int)

df["day"] = df["timestamp"].dt.day

df["month"] = df["timestamp"].dt.month

In [23]:
# ==========================================
# Time Since Previous Transaction
# ==========================================

df["previous_timestamp"] = (
    df.groupby("user_id")["timestamp"].shift(1)
)

df["seconds_since_previous"] = (
    df["timestamp"] - df["previous_timestamp"]
).dt.total_seconds()

df["rapid_transaction"] = (
    df["seconds_since_previous"] <= 300
).astype(int)

In [24]:
print(
    df["transaction_count_24h"].describe()
)

count    999891.000000
mean          9.010473
std           9.252277
min           0.000000
25%           4.000000
50%           7.000000
75%          10.000000
max          56.000000
Name: transaction_count_24h, dtype: float64


In [25]:
print(
    df.groupby("is_fraud")["transaction_count_24h"]
      .agg(["mean", "median", "max"])
)

               mean  median   max
is_fraud                         
0.0        6.072637     6.0  20.0
1.0       15.864945    10.0  56.0


In [26]:
print(
    df[
        [
            "transaction_id",
            "user_id",
            "timestamp",
            "transaction_count_24h",
            "is_fraud"
        ]
    ].head(20).to_string(index=False)
)

transaction_id    user_id           timestamp  transaction_count_24h  is_fraud
  TXN000159811 USER000001 2026-01-01 16:33:06                   28.0       1.0
  TXN000702246 USER000001 2026-01-05 10:27:09                    2.0       0.0
  TXN000145983 USER000001 2026-01-08 11:51:41                    0.0       0.0
  TXN000784417 USER000001 2026-01-08 20:24:43                    7.0       0.0
  TXN000344740 USER000001 2026-01-13 21:18:14                    6.0       0.0
  TXN000210574 USER000001 2026-01-16 01:27:30                    3.0       0.0
  TXN000353285 USER000001 2026-01-21 03:53:22                   26.0       1.0
  TXN000160177 USER000001 2026-01-24 16:03:30                    2.0       0.0
  TXN000709730 USER000001 2026-01-29 11:35:15                    4.0       0.0
  TXN000779274 USER000001 2026-01-31 18:47:31                    2.0       0.0
  TXN000173723 USER000001 2026-02-01 05:39:03                    1.0       0.0
  TXN000291699 USER000001 2026-02-03 14:07:46       

In [27]:
print(df["transaction_count_24h"].describe())

count    999891.000000
mean          9.010473
std           9.252277
min           0.000000
25%           4.000000
50%           7.000000
75%          10.000000
max          56.000000
Name: transaction_count_24h, dtype: float64


In [28]:
print(
    df.groupby("is_fraud")["transaction_count_24h"]
      .agg(["mean", "median", "max"])
)

               mean  median   max
is_fraud                         
0.0        6.072637     6.0  20.0
1.0       15.864945    10.0  56.0


In [29]:
print(
    df[
        [
            "transaction_id",
            "user_id",
            "timestamp",
            "transaction_count_24h",
            "is_fraud"
        ]
    ].head(20).to_string(index=False)
)

transaction_id    user_id           timestamp  transaction_count_24h  is_fraud
  TXN000159811 USER000001 2026-01-01 16:33:06                   28.0       1.0
  TXN000702246 USER000001 2026-01-05 10:27:09                    2.0       0.0
  TXN000145983 USER000001 2026-01-08 11:51:41                    0.0       0.0
  TXN000784417 USER000001 2026-01-08 20:24:43                    7.0       0.0
  TXN000344740 USER000001 2026-01-13 21:18:14                    6.0       0.0
  TXN000210574 USER000001 2026-01-16 01:27:30                    3.0       0.0
  TXN000353285 USER000001 2026-01-21 03:53:22                   26.0       1.0
  TXN000160177 USER000001 2026-01-24 16:03:30                    2.0       0.0
  TXN000709730 USER000001 2026-01-29 11:35:15                    4.0       0.0
  TXN000779274 USER000001 2026-01-31 18:47:31                    2.0       0.0
  TXN000173723 USER000001 2026-02-01 05:39:03                    1.0       0.0
  TXN000291699 USER000001 2026-02-03 14:07:46       

In [30]:
# ==========================================
# Historical User Spending Features
# ==========================================

df["user_previous_avg_amount"] = (
    df.groupby("user_id")["amount"]
      .transform(
          lambda x: x.shift(1).expanding().mean()
      )
)

In [31]:
df["user_previous_std_amount"] = (
    df.groupby("user_id")["amount"]
      .transform(
          lambda x: x.shift(1).expanding().std()
      )
)

In [32]:
# ==========================================
# Amount Deviation Ratio
# ==========================================

df["amount_deviation_ratio"] = (
    df["amount"] /
    (df["user_previous_avg_amount"] + 1e-6)
)

In [33]:
# ==========================================
# Historical Amount Z-Score
# ==========================================

df["amount_zscore"] = (
    (
        df["amount"] -
        df["user_previous_avg_amount"]
    )
    /
    (df["user_previous_std_amount"] + 1e-6)
)

df["amount_zscore"] = (
    df["amount_zscore"]
    .replace([np.inf, -np.inf], np.nan)
)

In [34]:
print(
    df[
        [
            "user_id",
            "timestamp",
            "amount",
            "user_previous_avg_amount",
            "user_previous_std_amount",
            "amount_deviation_ratio",
            "amount_zscore",
            "is_fraud"
        ]
    ].head(30).to_string(index=False)
)

   user_id           timestamp   amount  user_previous_avg_amount  user_previous_std_amount  amount_deviation_ratio  amount_zscore  is_fraud
USER000001 2026-01-01 16:33:06  3417.22                       NaN                       NaN                     NaN            NaN       1.0
USER000001 2026-01-05 10:27:09  3905.54               3417.220000                       NaN                1.142900            NaN       0.0
USER000001 2026-01-08 11:51:41   813.33               3661.380000                345.294383                0.222138      -8.248179       0.0
USER000001 2026-01-08 20:24:43  4776.32               2712.030000               1662.350857                1.761160       1.241790       0.0
USER000001 2026-01-13 21:18:14  2740.21               3228.102500               1705.167699                0.848861      -0.286126       0.0
USER000001 2026-01-16 01:27:30   934.21               3130.524000               1492.750977                0.298420      -1.471320       0.0
USER000001 20

In [35]:
print(
    df.groupby("is_fraud")[
        [
            "amount",
            "amount_deviation_ratio",
            "amount_zscore"
        ]
    ].agg(["mean", "median"])
)

                amount           amount_deviation_ratio            \
                  mean    median                   mean    median   
is_fraud                                                            
0.0        2628.373245  1830.920               0.404012  0.294491   
1.0       21528.875926  4970.465               3.374060  0.767227   

          amount_zscore            
                   mean    median  
is_fraud                           
0.0      -351985.699757 -0.346098  
1.0            2.984832 -0.141197  


In [36]:
# ==========================================
# User Transaction History Depth
# ==========================================

df["user_transaction_history"] = (
    df.groupby("user_id").cumcount()
)

print(
    df[
        [
            "user_id",
            "timestamp",
            "user_transaction_history",
            "user_previous_avg_amount",
            "amount_deviation_ratio",
            "amount_zscore"
        ]
    ].head(20).to_string(index=False)
)

   user_id           timestamp  user_transaction_history  user_previous_avg_amount  amount_deviation_ratio  amount_zscore
USER000001 2026-01-01 16:33:06                       0.0                       NaN                     NaN            NaN
USER000001 2026-01-05 10:27:09                       1.0               3417.220000                1.142900            NaN
USER000001 2026-01-08 11:51:41                       2.0               3661.380000                0.222138      -8.248179
USER000001 2026-01-08 20:24:43                       3.0               2712.030000                1.761160       1.241790
USER000001 2026-01-13 21:18:14                       4.0               3228.102500                0.848861      -0.286126
USER000001 2026-01-16 01:27:30                       5.0               3130.524000                0.298420      -1.471320
USER000001 2026-01-21 03:53:22                       6.0               2764.471667               16.561327      26.748121
USER000001 2026-01-24 16

In [37]:
# ==========================================
# Device Behavior
# ==========================================

device_user_count = (
    df.groupby("device_id")["user_id"]
      .nunique()
      .rename("device_user_count")
)

df = df.merge(
    device_user_count,
    on="device_id",
    how="left"
)

In [38]:
device_transaction_count = (
    df.groupby("device_id")
      .size()
      .rename("device_transaction_count")
)

df = df.merge(
    device_transaction_count,
    on="device_id",
    how="left"
)

In [39]:
print(
    df[
        [
            "device_id",
            "user_id",
            "is_new_device",
            "device_user_count",
            "device_transaction_count",
            "is_fraud"
        ]
    ].head(20).to_string(index=False)
)

   device_id    user_id  is_new_device  device_user_count  device_transaction_count  is_fraud
DEVICE010878 USER000001            0.0                  6                        42       1.0
DEVICE004548 USER000001            0.0                 12                        75       0.0
DEVICE010878 USER000001            0.0                  6                        42       0.0
DEVICE004548 USER000001            0.0                 12                        75       0.0
DEVICE004548 USER000001            0.0                 12                        75       0.0
DEVICE010878 USER000001            0.0                  6                        42       0.0
DEVICE014312 USER000001            1.0                 11                        40       1.0
DEVICE004548 USER000001            0.0                 12                        75       0.0
DEVICE004548 USER000001            0.0                 12                        75       0.0
DEVICE010878 USER000001            0.0                  6   

In [40]:
# ==========================================
# Location Behavior
# ==========================================

location_frequency = (
    df.groupby("location")
      .size()
      .rename("location_transaction_count")
)

df = df.merge(
    location_frequency,
    on="location",
    how="left"
)

In [41]:
user_location_frequency = (
    df.groupby(["user_id", "location"])
      .size()
      .rename("user_location_transaction_count")
      .reset_index()
)

df = df.merge(
    user_location_frequency,
    on=["user_id", "location"],
    how="left"
)

In [42]:
df["long_distance_transaction"] = (
    df["distance_from_home"] > 500
).astype(int)

In [43]:
print(
    df[
        [
            "user_id",
            "location",
            "distance_from_home",
            "is_new_location",
            "location_transaction_count",
            "user_location_transaction_count",
            "long_distance_transaction",
            "is_fraud"
        ]
    ].head(20).to_string(index=False)
)

   user_id  location  distance_from_home  is_new_location  location_transaction_count  user_location_transaction_count  long_distance_transaction  is_fraud
USER000001     Kochi                12.1              0.0                       97524                             70.0                          0       1.0
USER000001     Kochi                64.7              0.0                       97524                             70.0                          0       0.0
USER000001     Kochi                71.5              0.0                       97524                             70.0                          0       0.0
USER000001     Kochi                73.0              0.0                       97524                             70.0                          0       0.0
USER000001     Kochi                88.3              0.0                       97524                             70.0                          0       0.0
USER000001     Kochi                31.8              0.0       

In [44]:
print(
    df.groupby("is_fraud")[
        [
            "distance_from_home",
            "long_distance_transaction",
            "is_new_location"
        ]
    ].agg(["mean", "median", "max"])
)

         distance_from_home                long_distance_transaction         \
                       mean median     max                      mean median   
is_fraud                                                                      
0.0              140.733040   53.9  2500.0                  0.060429    0.0   
1.0              685.215552   89.9  2500.0                  0.436783    0.0   

             is_new_location              
         max            mean median  max  
is_fraud                                  
0.0        1        0.000009    0.0  1.0  
1.0        1        0.400800    0.0  1.0  


In [45]:
# ==========================================
# Merchant Behavior
# ==========================================

user_merchant_frequency = (
    df.groupby(["user_id", "merchant_id"])
      .size()
      .rename("user_merchant_transaction_count")
      .reset_index()
)

df = df.merge(
    user_merchant_frequency,
    on=["user_id", "merchant_id"],
    how="left"
)

In [46]:
user_category_frequency = (
    df.groupby(["user_id", "merchant_category"])
      .size()
      .rename("user_category_transaction_count")
      .reset_index()
)

df = df.merge(
    user_category_frequency,
    on=["user_id", "merchant_category"],
    how="left"
)

In [47]:
df["new_merchant_for_user"] = (
    df["user_merchant_transaction_count"] == 1
).astype(int)

In [48]:
print(
    df[
        [
            "user_id",
            "merchant_id",
            "merchant_category",
            "user_merchant_transaction_count",
            "user_category_transaction_count",
            "new_merchant_for_user",
            "is_fraud"
        ]
    ].head(20).to_string(index=False)
)

   user_id    merchant_id merchant_category  user_merchant_transaction_count  user_category_transaction_count  new_merchant_for_user  is_fraud
USER000001 MERCHANT003264           grocery                              1.0                              6.0                      1       1.0
USER000001 MERCHANT001983              fuel                              1.0                              5.0                      1       0.0
USER000001 MERCHANT001089              fuel                              1.0                              5.0                      1       0.0
USER000001 MERCHANT001289            travel                              1.0                             11.0                      1       0.0
USER000001 MERCHANT004589            retail                              1.0                             15.0                      1       0.0
USER000001 MERCHANT003452        restaurant                              1.0                              7.0                      1       0.0

In [49]:
print(
    df.groupby("is_fraud")[
        [
            "user_merchant_transaction_count",
            "user_category_transaction_count",
            "new_merchant_for_user"
        ]
    ].agg(["mean", "median", "max"])
)

         user_merchant_transaction_count              \
                                    mean median  max   
is_fraud                                               
0.0                             1.016847    1.0  3.0   
1.0                             1.016935    1.0  3.0   

         user_category_transaction_count              new_merchant_for_user  \
                                    mean median   max                  mean   
is_fraud                                                                      
0.0                             9.317784    9.0  25.0              0.983178   
1.0                             9.967344   10.0  25.0              0.983056   

                     
         median max  
is_fraud             
0.0         1.0   1  
1.0         1.0   1  


In [50]:
print("Minimum:", df["timestamp"].min())
print("Maximum:", df["timestamp"].max())

Minimum: 2026-01-01 00:00:02
Maximum: 2026-09-30 23:59:53


In [51]:
print(
    df["timestamp"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
)

timestamp
2026-01    113371
2026-02    102696
2026-03    113265
2026-04    109979
2026-05    113865
2026-06    109965
2026-07    113491
2026-08    113448
2026-09    109811
Freq: M, Name: count, dtype: int64


In [ ]:
# ==========================================
# TIME-BASED DATA SPLIT
# ==========================================

train_df = df[df["timestamp"] < "2026-08-01"].copy()

val_df = df[
    (df["timestamp"] >= "2026-08-01") &
    (df["timestamp"] < "2026-09-01")
].copy()

test_df = df[
    df["timestamp"] >= "2026-09-01"
].copy()

print("TRAIN:", train_df.shape)
print("VALIDATION:", val_df.shape)
print("TEST:", test_df.shape)

TRAIN: (776632, 47)
VALIDATION: (113448, 47)
TEST: (109811, 47)


In [53]:
print("\nFraud distribution:")
print("TRAIN")
print(train_df["is_fraud"].value_counts(dropna=False))
print("\nVALIDATION")
print(val_df["is_fraud"].value_counts(dropna=False))
print("\nTEST")
print(test_df["is_fraud"].value_counts(dropna=False))


Fraud distribution:
TRAIN
is_fraud
0.0    543644
1.0    232988
Name: count, dtype: int64

VALIDATION
is_fraud
0.0    79347
1.0    34101
Name: count, dtype: int64

TEST
is_fraud
0.0    76918
1.0    32893
Name: count, dtype: int64


In [54]:
for name, data in [
    ("TRAIN", train_df),
    ("VALIDATION", val_df),
    ("TEST", test_df)
]:
    fraud_rate = data["is_fraud"].mean() * 100
    print(f"{name}: {fraud_rate:.2f}% fraud")

TRAIN: 30.00% fraud
VALIDATION: 30.06% fraud
TEST: 29.95% fraud


In [56]:
# ==========================================
# INITIAL MODEL FEATURE SET
# ==========================================

drop_columns = [
    "transaction_id",
    "id",
    "user_id",
    "merchant_id",
    "device_id",
    "ip_address",
    "timestamp",
    "previous_timestamp",

    # Exploratory full-dataset features
    "device_user_count",
    "device_transaction_count",
    "location_transaction_count",
    "user_location_transaction_count",
    "user_merchant_transaction_count",
    "user_category_transaction_count",
    "new_merchant_for_user"
]

model_df = df.drop(
    columns=[col for col in drop_columns if col in df.columns]
).copy()

print("Model dataset shape:", model_df.shape)
print("\nColumns:")
print(model_df.columns.tolist())

Model dataset shape: (999891, 33)

Columns:
['amount', 'merchant_category', 'location', 'payment_method', 'account_age', 'transaction_count_24h', 'avg_transaction_amount', 'distance_from_home', 'is_new_device', 'is_new_location', 'is_fraud', 'flag_missing_amount', 'flag_negative_amount', 'flag_missing_user', 'flag_missing_merchant', 'flag_negative_distance', 'flag_duplicate_transaction', 'flag_invalid_timestamp', 'flag_unknown_category', 'data_quality_issue', 'hour', 'day_of_week', 'is_weekend', 'day', 'month', 'seconds_since_previous', 'rapid_transaction', 'user_previous_avg_amount', 'user_previous_std_amount', 'amount_deviation_ratio', 'amount_zscore', 'user_transaction_history', 'long_distance_transaction']


In [58]:
categorical_cols = [
    "merchant_category",
    "location",
    "payment_method"
]

for col in categorical_cols:
    print(f"\n{col}")
    print("Unique:", X_train[col].nunique())
    print(X_train[col].value_counts().head(10))


merchant_category
Unique: 11
merchant_category
entertainment    97596
travel           94578
electronics      92951
healthcare       73019
utilities        71648
clothing         70591
grocery          70461
restaurant       70408
retail           68347
fuel             66942
Name: count, dtype: int64

location
Unique: 11
location
Thiruvananthapuram    80074
Hyderabad             79852
Chennai               79082
Kozhikode             78758
Pune                  78111
Mumbai                77059
Kolkata               76627
Kochi                 75784
Bengaluru             75708
Delhi                 75498
Name: count, dtype: int64

payment_method
Unique: 4
payment_method
upi            388510
card           232994
net_banking    116529
wallet          38599
Name: count, dtype: int64


In [59]:
print("\nMissing values in X_train:")
print(X_train.isna().sum().sort_values(ascending=False).head(15))



Missing values in X_train:
amount_zscore               24230
user_previous_std_amount    24074
amount_deviation_ratio      12230
user_previous_avg_amount    12070
seconds_since_previous      12067
amount                        163
user_transaction_history       67
distance_from_home             63
is_new_device                  14
is_new_location                 8
avg_transaction_amount          0
transaction_count_24h           0
payment_method                  0
account_age                     0
location                        0
dtype: int64


In [62]:
categorical_cols = [
    "merchant_category",
    "location",
    "payment_method"
]

for col in categorical_cols:
    print(f"\n{col}")
    print("Unique:", X_train[col].nunique())
    print(X_train[col].value_counts().head(10))


merchant_category
Unique: 11
merchant_category
entertainment    97596
travel           94578
electronics      92951
healthcare       73019
utilities        71648
clothing         70591
grocery          70461
restaurant       70408
retail           68347
fuel             66942
Name: count, dtype: int64

location
Unique: 11
location
Thiruvananthapuram    80074
Hyderabad             79852
Chennai               79082
Kozhikode             78758
Pune                  78111
Mumbai                77059
Kolkata               76627
Kochi                 75784
Bengaluru             75708
Delhi                 75498
Name: count, dtype: int64

payment_method
Unique: 4
payment_method
upi            388510
card           232994
net_banking    116529
wallet          38599
Name: count, dtype: int64
